# Bangladesh Urban Center Mapping — Earth Search Tutorial Style + Multi-Factor Workflow

This notebook keeps the **interactive/tutorial style** of an Earth Search STAC notebook, but redesigns it for a **national Bangladesh urban-center mapping project**.

**Workflow:** Bangladesh AOI → Earth Search Sentinel-2 → footprints → resilient stack → SCL masking → median composite → NDBI/NDVI/MNDWI/BSI → built-up probability → VIIRS → population → roads → POIs → DEM/slope → weighted Urban Score → threshold → connected components → final urban-center polygons.

Sentinel-2 is searched online automatically. National VIIRS, population, DEM, roads and POI layers are loaded from local files so the workflow remains reproducible and does not depend on huge one-shot web requests.


In [1]:
# 0. OPTIONAL INSTALLATION
# Recommended: use one conda-forge environment only.
#
# conda install -c conda-forge --override-channels ^
#     python=3.11 geopandas rasterio pyproj rioxarray xarray dask ^
#     pystac-client stackstac leafmap shapely scipy scikit-image ^
#     matplotlib pandas numpy fiona pyogrio


In [2]:
# 1. CONFIGURATION
from pathlib import Path
import os

PROJECT_DIR = Path(r"E:\Geospatial\Urban Center\Urban-Center")
AOI_PATH = PROJECT_DIR / "bgd_admin_boundaries.shp" / "bgd_admin0.shp"
OUTPUT_DIR = PROJECT_DIR / "outputs"
INPUT_DIR = PROJECT_DIR / "inputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_RANGE = "2025-01-01/2025-03-31"
MAX_CLOUD = 10
N_BEST_PER_TILE = 5

TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

WEIGHTS = {
    "builtup": 0.30,
    "nightlight": 0.20,
    "population": 0.20,
    "road": 0.10,
    "poi": 0.10,
    "topography": 0.10,
}

URBAN_SCORE_THRESHOLD = 0.55
MIN_PATCH_AREA_KM2 = 1.0
MIN_MEAN_POP_DENSITY = 500.0
MIN_MEAN_URBAN_SCORE = 0.60

VIIRS_PATH = INPUT_DIR / "viirs_2025.tif"
POPULATION_PATH = INPUT_DIR / "population_density_2025.tif"
DEM_PATH = INPUT_DIR / "dem_bangladesh.tif"
ROADS_PATH = INPUT_DIR / "roads_bangladesh.gpkg"
POI_PATH = INPUT_DIR / "poi_bangladesh.gpkg"

os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["GDAL_HTTP_RETRY_CODES"] = "ALL"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

print("Project:", PROJECT_DIR)
print("AOI:", AOI_PATH)
print("Outputs:", OUTPUT_DIR)


Project: E:\Geospatial\Urban Center\Urban-Center
AOI: E:\Geospatial\Urban Center\Urban-Center\bgd_admin_boundaries.shp\bgd_admin0.shp
Outputs: E:\Geospatial\Urban Center\Urban-Center\outputs


In [3]:
# 2. IMPORTS + ENVIRONMENT CHECK
from collections import defaultdict
import warnings

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling, MergeAlg
from rasterio.features import rasterize, shapes
from rasterio.errors import RasterioIOError
import pyproj
from shapely.geometry import shape
from shapely.ops import unary_union
import xarray as xr
import rioxarray
import stackstac
from pystac_client import Client
from scipy import ndimage
import leafmap

warnings.filterwarnings("ignore", category=FutureWarning)

print("Rasterio:", rasterio.__version__)
print("GDAL:", rasterio.__gdal_version__)
print("PROJ:", pyproj.proj_version_str)
print("StackSTAC:", stackstac.__version__)

try:
    print("CRS test:", rasterio.crs.CRS.from_epsg(TARGET_EPSG))
except Exception as exc:
    raise RuntimeError(
        "GDAL/PROJ environment is inconsistent. Repair/recreate the conda "
        f"environment before continuing. Original error: {exc}"
    )


Rasterio: 1.4.4
GDAL: 3.10.3
PROJ: 9.5.1
StackSTAC: 0.5.1


RuntimeError: GDAL/PROJ environment is inconsistent. Repair/recreate the conda environment before continuing. Original error: The EPSG code is unknown. PROJ: proj_create_from_database: C:\Users\HP\.conda\envs\geo\Library\share\proj\proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 5 is expected. It comes from another PROJ installation.

In [4]:
# 3. LOAD BANGLADESH AOI
if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI not found: {AOI_PATH}\nEdit PROJECT_DIR/AOI_PATH in Cell 1."
    )

bd = gpd.read_file(AOI_PATH)

if bd.empty:
    raise ValueError("AOI contains no features.")
if bd.crs is None:
    raise ValueError("AOI has no CRS.")

bd = bd.to_crs(4326)
aoi = bd[["geometry"]].dissolve().reset_index(drop=True)

if not aoi.geometry.iloc[0].is_valid:
    aoi["geometry"] = aoi.geometry.buffer(0)

bd_geom = aoi.geometry.iloc[0]
west, south, east, north = map(float, aoi.total_bounds)

print("AOI CRS:", aoi.crs)
print("Bounds:", (west, south, east, north))
print("AOI valid:", bd_geom.is_valid)


AOI CRS: EPSG:4326
Bounds: (88.00816912400006, 20.590608254000188, 92.68005782300008, 26.634548266000024)
AOI valid: True


In [5]:
# 4. INTERACTIVE BANGLADESH MAP
m = leafmap.Map(center=[23.6850, 90.3563], zoom=7, height="700px")

m.add_gdf(
    aoi,
    layer_name="Bangladesh AOI",
    style={
        "color": "red",
        "weight": 2,
        "fillColor": "red",
        "fillOpacity": 0.05,
    },
)

m


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [6]:
# 5. EARTH SEARCH STAC — WHOLE BANGLADESH
# Use bbox in the API request to avoid "request entity too large".
# Exact Bangladesh intersection is done locally afterwards.

catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=[west, south, east, north],
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
)

items_raw = list(search.items())
print("Raw bbox results:", len(items_raw))

items_bd = []
for item in items_raw:
    if item.geometry is None:
        continue
    try:
        if shape(item.geometry).intersects(bd_geom):
            items_bd.append(item)
    except Exception as exc:
        print("Skipped:", item.id, type(exc).__name__)

if not items_bd:
    raise RuntimeError("No Sentinel-2 scenes intersect Bangladesh.")

dates = sorted({
    item.datetime.strftime("%Y-%m-%d")
    for item in items_bd
    if item.datetime is not None
})

print("Scenes intersecting Bangladesh:", len(items_bd))
print("Unique acquisition dates:", len(dates))
print("First dates:", dates[:10])


Raw bbox results: 984
Scenes intersecting Bangladesh: 634
Unique acquisition dates: 51
First dates: ['2025-01-02', '2025-01-03', '2025-01-05', '2025-01-07', '2025-01-10', '2025-01-12', '2025-01-13', '2025-01-15', '2025-01-17', '2025-01-18']


In [7]:
# 6. MAP ALL SENTINEL-2 SEARCH FOOTPRINTS
scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in items_bd
    ],
    crs="EPSG:4326",
)

m2 = leafmap.Map(center=[23.6850, 90.3563], zoom=7, height="700px")
m2.add_gdf(
    scene_gdf,
    layer_name="Sentinel-2 footprints",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.04,
    },
)
m2.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "red", "weight": 3, "fillOpacity": 0.0},
)
m2


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [8]:
# 7. LOW-CLOUD SCENE SELECTION + COVERAGE COMPLETION

def get_tile_id(item):
    parts = item.id.split("_")
    if len(parts) > 1 and parts[1].startswith("T"):
        return parts[1]
    grid_code = item.properties.get("grid:code")
    if grid_code:
        return str(grid_code)
    raise ValueError(f"Could not identify tile: {item.id}")


items_by_tile = defaultdict(list)
for item in items_bd:
    try:
        items_by_tile[get_tile_id(item)].append(item)
    except ValueError as exc:
        print(exc)

selected_items = []

for tile_id, tile_items in items_by_tile.items():
    tile_items = sorted(
        tile_items,
        key=lambda item: (
            item.properties.get("eo:cloud_cover", 100),
            item.datetime.isoformat() if item.datetime else "",
        ),
    )
    selected_items.extend(tile_items[:N_BEST_PER_TILE])


def footprint_union(stac_items):
    geoms = [
        shape(item.geometry)
        for item in stac_items
        if item.geometry is not None
    ]
    if not geoms:
        raise RuntimeError("No valid footprints.")
    return unary_union(geoms)


selected_union = footprint_union(selected_items)
missing_geom = bd_geom.difference(selected_union)
selected_ids = {item.id for item in selected_items}

remaining_items = sorted(
    [item for item in items_bd if item.id not in selected_ids],
    key=lambda item: item.properties.get("eo:cloud_cover", 100),
)

extra_items = []

for item in remaining_items:
    if missing_geom.is_empty:
        break

    scene_geom = shape(item.geometry)
    gain = missing_geom.intersection(scene_geom)

    if not gain.is_empty and gain.area > 0:
        selected_items.append(item)
        extra_items.append(item)
        selected_union = selected_union.union(scene_geom)
        missing_geom = bd_geom.difference(selected_union)

area_check = gpd.GeoDataFrame(
    {"type": ["AOI", "Covered"]},
    geometry=[bd_geom, bd_geom.intersection(selected_union)],
    crs="EPSG:4326",
).to_crs(TARGET_EPSG)

coverage_pct = (
    area_check.geometry.iloc[1].area
    / area_check.geometry.iloc[0].area
    * 100
)

print("Unique MGRS tiles:", len(items_by_tile))
print("Selected scenes:", len(selected_items))
print("Extra scenes added:", len(extra_items))
print(f"Bangladesh footprint coverage: {coverage_pct:.6f}%")


Unique MGRS tiles: 36
Selected scenes: 187
Extra scenes added: 7
Bangladesh footprint coverage: 100.000000%


In [9]:
# 8. MAP FINAL SELECTED SCENES
selected_scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in selected_items
    ],
    crs="EPSG:4326",
)

m3 = leafmap.Map(center=[23.6850, 90.3563], zoom=7, height="700px")
m3.add_gdf(
    selected_scene_gdf,
    layer_name="Selected scenes",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.08,
    },
)
m3.add_gdf(
    aoi,
    layer_name="Bangladesh boundary",
    style={"color": "red", "weight": 3, "fillOpacity": 0.0},
)
m3


Map(center=[23.685, 90.3563], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [10]:
# 9. BUILD ONE RESILIENT NATIONAL STACK
REQUIRED_ASSETS = ["blue", "green", "red", "nir", "swir16", "scl"]

selected_items_clean = [
    item for item in selected_items
    if all(asset in item.assets for asset in REQUIRED_ASSETS)
]

print("Scenes with all required assets:", len(selected_items_clean))
print(
    "Dropped for missing assets:",
    len(selected_items) - len(selected_items_clean),
)

if not selected_items_clean:
    raise RuntimeError("No selected scene contains all required assets.")

sentinel = stackstac.stack(
    selected_items_clean,
    assets=REQUIRED_ASSETS,
    bounds_latlon=[west, south, east, north],
    epsg=TARGET_EPSG,
    resolution=RESOLUTION_M,
    chunksize=CHUNK_SIZE,
    dtype=np.float32,
    fill_value=np.float32(np.nan),
    rescale=False,
    errors_as_nodata=(RasterioIOError(r".*"),),
)

print(sentinel)
print("Virtual stack shape:", sentinel.shape)


Scenes with all required assets: 187
Dropped for missing assets: 0
<xarray.DataArray 'stackstac-70774311adfebdce4b4b7202b37e8630' (time: 187,
                                                                band: 6,
                                                                y: 7078, x: 4509)> Size: 143GB
dask.array<fetch_raster_window, shape=(187, 6, 7078, 4509), dtype=float32, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>
Coordinates: (12/54)
  * time                                     (time) datetime64[ns] 1kB 2025-0...
    id                                       (time) <U30 22kB 'S2A_T45RYK_202...
  * band                                     (band) <U6 144B 'blue' ... 'scl'
  * x                                        (x) float64 36kB 8.492e+06 ... 8...
  * y                                        (y) float64 57kB 3.28e+06 ... 2....
    view:sun_elevation                       (time) float64 1kB 37.37 ... 62.77
    ...                                       ...
    title

In [11]:
# 10. SMALL REAL-READ TEST
# Test a central national window rather than the top-left bbox corner.

cx = sentinel.sizes["x"] // 2
cy = sentinel.sizes["y"] // 2

test_stack = sentinel.isel(
    time=slice(0, min(3, sentinel.sizes["time"])),
    x=slice(max(0, cx - 128), min(cx + 128, sentinel.sizes["x"])),
    y=slice(max(0, cy - 128), min(cy + 128, sentinel.sizes["y"])),
).compute()

print("Small remote-read test successful.")
print("Test shape:", test_stack.shape)
print("Finite values:", int(np.isfinite(test_stack.values).sum()))


Small remote-read test successful.
Test shape: (3, 6, 256, 256)
Finite values: 0


In [12]:
# 11. SCL CLOUD MASK + TEMPORAL MEDIAN
scl = sentinel.sel(band="scl")

INVALID_SCL = [0, 1, 3, 8, 9, 10, 11]
valid = ~scl.isin(INVALID_SCL)

blue_med = sentinel.sel(band="blue").where(valid).median("time", skipna=True)
green_med = sentinel.sel(band="green").where(valid).median("time", skipna=True)
red_med = sentinel.sel(band="red").where(valid).median("time", skipna=True)
nir_med = sentinel.sel(band="nir").where(valid).median("time", skipna=True)
swir_med = sentinel.sel(band="swir16").where(valid).median("time", skipna=True)

print("Cloud-masked median composites prepared.")


Cloud-masked median composites prepared.


In [13]:
# 12. NDBI, NDVI, MNDWI, BSI
EPS = np.float32(1e-6)

def safe_nd(a, b):
    den = a + b
    return (
        (a - b) / den.where(np.abs(den) > EPS)
    ).clip(-1, 1).astype("float32")

ndbi = safe_nd(swir_med, nir_med).rename("NDBI")
ndvi = safe_nd(nir_med, red_med).rename("NDVI")
mndwi = safe_nd(green_med, swir_med).rename("MNDWI")

bsi_num = (swir_med + red_med) - (nir_med + blue_med)
bsi_den = (swir_med + red_med) + (nir_med + blue_med)
bsi = (
    bsi_num / bsi_den.where(np.abs(bsi_den) > EPS)
).clip(-1, 1).astype("float32").rename("BSI")

print("Prepared NDBI, NDVI, MNDWI, BSI.")


Prepared NDBI, NDVI, MNDWI, BSI.


In [14]:
# 13. SENTINEL BUILT-UP PROBABILITY
# Interpretable project score, not a pretrained probability model.

def norm_index01(da):
    return ((da + 1.0) / 2.0).clip(0, 1).astype("float32")

ndbi01 = norm_index01(ndbi)
ndvi01 = norm_index01(ndvi)
mndwi01 = norm_index01(mndwi)
bsi01 = norm_index01(bsi)

builtup_probability = (
    0.50 * ndbi01
    + 0.20 * (1.0 - ndvi01)
    + 0.20 * (1.0 - mndwi01)
    + 0.10 * (1.0 - bsi01)
).clip(0, 1).astype("float32").rename("Builtup_Probability")

print("Built-up probability prepared.")


Built-up probability prepared.


In [15]:
# 14. COMMON 100-m GRID + EXACT BANGLADESH CLIP
template = (
    ndbi.astype("float32")
    .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
)

aoi_target = aoi.to_crs(TARGET_EPSG)

def clip_bd(da):
    da = (
        da.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )
    return da.rio.clip(
        aoi_target.geometry,
        aoi_target.crs,
        drop=True,
        all_touched=False,
    )

print("Common grid CRS:", template.rio.crs)
print("Common grid resolution:", template.rio.resolution())


Common grid CRS: PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","6933"]]
Common grid resolution: (100.0, -100.0)


In [16]:
# 15. ALIGN ANY INPUT RASTER TO THE SENTINEL TEMPLATE
def align_raster_to_template(
    raster_path,
    template_da,
    resampling=Resampling.bilinear,
):
    raster_path = Path(raster_path)
    if not raster_path.exists():
        raise FileNotFoundError(raster_path)

    src = rioxarray.open_rasterio(
        raster_path,
        masked=True,
        chunks=True,
    ).squeeze(drop=True)

    return src.rio.reproject_match(
        template_da,
        resampling=resampling,
    ).astype("float32")


In [17]:
# 16. ROBUST 0-1 NORMALIZATION
def robust_norm01(da, q_low=0.02, q_high=0.98):
    q = da.quantile(
        [q_low, q_high],
        dim=("y", "x"),
        skipna=True,
    ).compute()

    lo = float(q.sel(quantile=q_low))
    hi = float(q.sel(quantile=q_high))

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid normalization range: {lo}, {hi}")

    return ((da - lo) / (hi - lo)).clip(0, 1).astype("float32")


In [18]:
# 17. VIIRS NIGHT-LIGHT SCORE
if VIIRS_PATH.exists():
    viirs = align_raster_to_template(
        VIIRS_PATH, template, Resampling.bilinear
    ).clip(min=0)

    viirs_log = xr.apply_ufunc(np.log1p, viirs)
    nightlight_score = robust_norm01(
        viirs_log
    ).rename("NightLight_Score")
    print("VIIRS READY")
else:
    nightlight_score = None
    print("Missing:", VIIRS_PATH)


Missing: E:\Geospatial\Urban Center\Urban-Center\inputs\viirs_2025.tif


In [20]:
# 18. POPULATION SCORE
if POPULATION_PATH.exists():
    population_density = align_raster_to_template(
        POPULATION_PATH, template, Resampling.bilinear
    ).clip(min=0)

    population_score = robust_norm01(
        xr.apply_ufunc(np.log1p, population_density)
    ).rename("Population_Score")
    print("POPULATION READY")
else:
    population_density = None
    population_score = None
    print("Missing:", POPULATION_PATH)


Missing: E:\Geospatial\Urban Center\Urban-Center\inputs\population_density_2025.tif


In [21]:
# 19. VECTOR → 100-m LOCAL DENSITY RASTER
def vector_density_to_template(
    vector_path,
    template_da,
    geometry_mode="line",
):
    vector_path = Path(vector_path)
    if not vector_path.exists():
        raise FileNotFoundError(vector_path)

    gdf = gpd.read_file(vector_path)
    if gdf.empty:
        raise ValueError(f"No features in {vector_path}")

    gdf = gdf.to_crs(TARGET_EPSG)
    gdf = gpd.clip(gdf, aoi_target)

    transform = template_da.rio.transform()
    out_shape = (
        template_da.sizes["y"],
        template_da.sizes["x"],
    )

    pairs = [
        (geom, 1)
        for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]

    if geometry_mode == "line":
        arr = rasterize(
            pairs,
            out_shape=out_shape,
            transform=transform,
            fill=0,
            dtype="float32",
            all_touched=True,
        )
    elif geometry_mode == "point":
        arr = rasterize(
            pairs,
            out_shape=out_shape,
            transform=transform,
            fill=0,
            dtype="float32",
            merge_alg=MergeAlg.add,
            all_touched=True,
        )
    else:
        raise ValueError("geometry_mode must be 'line' or 'point'")

    # Approx. 1-km neighborhood on a 100-m grid.
    window_cells = max(1, int(round(1000 / RESOLUTION_M)))
    local_density = ndimage.uniform_filter(
        arr,
        size=window_cells,
        mode="constant",
        cval=0,
    )

    da = xr.DataArray(
        local_density,
        dims=("y", "x"),
        coords={
            "y": template_da.y.values,
            "x": template_da.x.values,
        },
    ).astype("float32")

    return (
        da.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )


In [22]:
# 20. ROAD SCORE
if ROADS_PATH.exists():
    road_density = vector_density_to_template(
        ROADS_PATH, template, "line"
    )
    road_score = robust_norm01(
        road_density
    ).rename("Road_Score")
    print("ROADS READY")
else:
    road_density = None
    road_score = None
    print("Missing:", ROADS_PATH)


Missing: E:\Geospatial\Urban Center\Urban-Center\inputs\roads_bangladesh.gpkg


In [23]:
# 21. POI / SERVICE SCORE
if POI_PATH.exists():
    poi_density = vector_density_to_template(
        POI_PATH, template, "point"
    )
    poi_score = robust_norm01(
        poi_density
    ).rename("POI_Service_Score")
    print("POI READY")
else:
    poi_density = None
    poi_score = None
    print("Missing:", POI_PATH)


Missing: E:\Geospatial\Urban Center\Urban-Center\inputs\poi_bangladesh.gpkg


In [24]:
# 22. DEM + SLOPE + TOPOGRAPHY SCORE
if DEM_PATH.exists():
    dem = align_raster_to_template(
        DEM_PATH, template, Resampling.bilinear
    )

    dem_np = dem.compute().values.astype("float32")
    gy, gx = np.gradient(
        dem_np,
        RESOLUTION_M,
        RESOLUTION_M,
    )
    slope_np = np.degrees(
        np.arctan(np.sqrt(gx**2 + gy**2))
    ).astype("float32")

    slope = xr.DataArray(
        slope_np,
        dims=("y", "x"),
        coords={
            "y": template.y.values,
            "x": template.x.values,
        },
        name="Slope",
    )
    slope = (
        slope.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

    elevation01 = robust_norm01(dem)
    slope01 = robust_norm01(slope)

    topography_score = (
        0.50 * (1.0 - elevation01)
        + 0.50 * (1.0 - slope01)
    ).clip(0, 1).rename("Topography_Score")

    print("TOPOGRAPHY READY")
else:
    dem = None
    slope = None
    topography_score = None
    print("Missing:", DEM_PATH)


Missing: E:\Geospatial\Urban Center\Urban-Center\inputs\dem_bangladesh.tif


In [25]:
# 23. FACTOR READINESS
factors = {
    "builtup": builtup_probability,
    "nightlight": nightlight_score,
    "population": population_score,
    "road": road_score,
    "poi": poi_score,
    "topography": topography_score,
}

for name, value in factors.items():
    print(f"{name:12s}:", "READY" if value is not None else "MISSING")


builtup     : READY
nightlight  : MISSING
population  : MISSING
road        : MISSING
poi         : MISSING
topography  : MISSING


In [26]:
# 24. FINAL MULTI-FACTOR URBAN SCORE
missing = [
    name for name, value in factors.items()
    if value is None
]

if missing:
    raise RuntimeError(
        "Missing factors: " + ", ".join(missing)
        + ". Add the required input files and rerun their cells."
    )

factor_bd = {
    name: clip_bd(value)
    for name, value in factors.items()
}

urban_score = sum(
    WEIGHTS[name] * factor_bd[name]
    for name in WEIGHTS
).clip(0, 1).astype("float32").rename("Urban_Score")

print("Urban Score prepared.")


RuntimeError: Missing factors: nightlight, population, road, poi, topography. Add the required input files and rerun their cells.

In [ ]:
# 25. URBAN CENTER MASK
urban_mask = (
    urban_score >= URBAN_SCORE_THRESHOLD
).astype("uint8").rename("Urban_Center_Mask")

print("Threshold:", URBAN_SCORE_THRESHOLD)


In [ ]:
# 26. CONNECTED COMPONENTS + MINIMUM PATCH FILTER
mask_np = urban_mask.compute().values.astype(bool)

labels, n_components = ndimage.label(mask_np)

pixel_area_km2 = (
    RESOLUTION_M * RESOLUTION_M
) / 1_000_000.0

min_pixels = max(
    1,
    int(np.ceil(
        MIN_PATCH_AREA_KM2 / pixel_area_km2
    )),
)

counts = np.bincount(labels.ravel())
keep_labels = np.where(counts >= min_pixels)[0]
keep_labels = keep_labels[keep_labels != 0]

filtered_np = np.isin(labels, keep_labels)

filtered_mask = xr.DataArray(
    filtered_np.astype("uint8"),
    dims=("y", "x"),
    coords={
        "y": urban_mask.y.values,
        "x": urban_mask.x.values,
    },
    name="Urban_Center_Filtered",
)

filtered_mask = (
    filtered_mask.rio
    .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
)

print("Initial components:", n_components)
print("Retained components:", len(keep_labels))


In [ ]:
# 27. POLYGONIZE
records = []

for geom, value in shapes(
    filtered_mask.values,
    mask=filtered_mask.values.astype(bool),
    transform=filtered_mask.rio.transform(),
):
    if int(value) == 1:
        records.append({
            "geometry": shape(geom),
            "class": 1,
        })

urban_polygons = gpd.GeoDataFrame(
    records,
    crs=f"EPSG:{TARGET_EPSG}",
)

if not urban_polygons.empty:
    urban_polygons["area_km2"] = (
        urban_polygons.geometry.area / 1_000_000.0
    )
    urban_polygons = urban_polygons.reset_index(drop=True)
    urban_polygons["urban_id"] = np.arange(
        1, len(urban_polygons) + 1
    )

print("Urban polygons:", len(urban_polygons))


In [ ]:
# 28. POLYGON MEAN VALUES
def zonal_mean(gdf, da, field_name):
    arr = da.compute().values
    transform = da.rio.transform()
    means = []

    for geom in gdf.geometry:
        zone = rasterize(
            [(geom, 1)],
            out_shape=arr.shape,
            transform=transform,
            fill=0,
            dtype="uint8",
        ).astype(bool)

        vals = arr[zone]
        vals = vals[np.isfinite(vals)]
        means.append(
            float(vals.mean()) if vals.size else np.nan
        )

    gdf[field_name] = means
    return gdf


if not urban_polygons.empty:
    urban_polygons = zonal_mean(
        urban_polygons,
        urban_score,
        "mean_score",
    )
    urban_polygons = zonal_mean(
        urban_polygons,
        population_density,
        "mean_popden",
    )


In [ ]:
# 29. FINAL FILTERING
if urban_polygons.empty:
    final_urban_centers = urban_polygons.copy()
else:
    final_urban_centers = urban_polygons[
        (urban_polygons["area_km2"] >= MIN_PATCH_AREA_KM2)
        & (urban_polygons["mean_score"] >= MIN_MEAN_URBAN_SCORE)
        & (urban_polygons["mean_popden"] >= MIN_MEAN_POP_DENSITY)
    ].copy()

    final_urban_centers = final_urban_centers.reset_index(drop=True)
    final_urban_centers["urban_id"] = np.arange(
        1, len(final_urban_centers) + 1
    )

print("Final urban centers:", len(final_urban_centers))


In [ ]:
# 30. EXPORT
raster_outputs = {
    "NDBI_2025_Q1_100m.tif": clip_bd(ndbi),
    "NDVI_2025_Q1_100m.tif": clip_bd(ndvi),
    "MNDWI_2025_Q1_100m.tif": clip_bd(mndwi),
    "BSI_2025_Q1_100m.tif": clip_bd(bsi),
    "Builtup_Probability_2025_Q1_100m.tif": clip_bd(builtup_probability),
    "Urban_Score_2025_100m.tif": urban_score,
    "Urban_Center_Mask_2025_100m.tif": filtered_mask,
}

for filename, da in raster_outputs.items():
    path = OUTPUT_DIR / filename
    print("Writing:", path)
    da.rio.to_raster(
        path,
        compress="DEFLATE",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

gpkg_path = OUTPUT_DIR / "Bangladesh_Urban_Centers_2025.gpkg"

final_urban_centers.to_file(
    gpkg_path,
    layer="urban_centers",
    driver="GPKG",
)

print("Finished:", gpkg_path)


In [ ]:
# 31. FINAL INTERACTIVE MAP
final_map = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px",
)

final_map.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={
        "color": "black",
        "weight": 2,
        "fillOpacity": 0,
    },
)

if not final_urban_centers.empty:
    final_map.add_gdf(
        final_urban_centers.to_crs(4326),
        layer_name="Final Urban Centers",
        style={
            "color": "red",
            "weight": 1,
            "fillColor": "red",
            "fillOpacity": 0.45,
        },
    )

final_map


## Required local files for the full multi-factor model

Put these under:

`E:\Geospatial\Urban Center\Urban-Center\inputs`

- `viirs_2025.tif`
- `population_density_2025.tif`
- `dem_bangladesh.tif`
- `roads_bangladesh.gpkg`
- `poi_bangladesh.gpkg`

The Sentinel-2 section is fully online through Earth Search and covers the whole Bangladesh AOI rather than one tutorial point/tile.
